In [ ]:
import math
import re
import os
import pandas as pd
from collections import Counter
from groq import Groq

print('Imports OK')
print(f'pandas: {pd.__version__}')

In [11]:
CORPUS = {
    'boas_vindas': {
        'padroes': [
            'oi', 'ola', 'hey', 'hello', 'bom dia', 'boa tarde',
            'boa noite', 'e ai', 'tudo bem', 'oi tudo bem', 'oi bot'
        ]
    },
    'despedida': {
        'padroes': [
            'tchau', 'ate logo', 'xau', 'bye', 'adeus',
            'ate mais', 'encerrar', 'sair', 'obrigado tchau'
        ]
    },
    'tema_geral': {
        'padroes': [
            'o que voce sabe', 'sobre o que fala', 'quais temas',
            'me ajude', 'quais assuntos', 'o que e formula 1',
            'me fale sobre f1', 'o que e f1'
        ]
    },
    'piloto': {
        'padroes': [
            'quem e verstappen', 'fale sobre hamilton', 'quem pilota pela red bull',
            'quem e leclerc', 'pilotos da f1', 'melhores pilotos',
            'quem sao os pilotos', 'quem e o melhor piloto',
            'piloto mais vitorioso', 'quantos titulos hamilton tem', 'fale sobre norris'
        ]
    },
    'equipe': {
        'padroes': [
            'quais sao as equipes', 'fale sobre red bull', 'fale sobre ferrari',
            'fale sobre mercedes', 'equipes da f1', 'construtores',
            'campeonato de construtores', 'melhor equipe', 'equipe mais vitoriosa'
        ]
    },
    'corrida': {
        'padroes': [
            'quando e o proximo gp', 'calendario f1', 'quantas corridas tem',
            'resultado da corrida', 'quem ganhou', 'gp do brasil',
            'monaco', 'silverstone', 'monza', 'circuito mais rapido'
        ]
    },
    'campeonato': {
        'padroes': [
            'classificacao', 'quem lidera', 'lider do campeonato',
            'pontuacao', 'tabela de pontos', 'standings',
            'quem esta na frente', 'campeonato 2024'
        ]
    },
    'consulta_csv': {
        'padroes': [
            'pontuacao dos pilotos', 'mostrar classificacao', 'tabela campeonato',
            'dados de verstappen', 'dados de hamilton', 'dados de leclerc',
            'dados de norris', 'dados de sainz', 'dados de russell',
            'dados de perez', 'quem tem mais pontos', 'top pilotos'
        ]
    }
}

print(f'Corpus: {len(CORPUS)} intencoes')
for k, v in CORPUS.items():
    print(f'  {k}: {len(v["padroes"])} padroes')

Corpus: 8 intencoes
  boas_vindas: 11 padroes
  despedida: 9 padroes
  tema_geral: 8 padroes
  piloto: 11 padroes
  equipe: 9 padroes
  corrida: 10 padroes
  campeonato: 8 padroes
  consulta_csv: 12 padroes


In [10]:
def normalizar(texto):
    """Remove acentos simples, minusculo, sem pontuacao."""
    texto = texto.lower().strip()
    substituicoes = {
        'á':'a','à':'a','ã':'a','â':'a','é':'e','ê':'e',
        'í':'i','ó':'o','ô':'o','õ':'o','ú':'u','ç':'c'
    }
    for orig, sub in substituicoes.items():
        texto = texto.replace(orig, sub)
    return re.sub(r'[^a-z0-9 ]', '', texto)


def levenshtein(s1, s2):
    """Retorna similaridade entre 0 e 1 (1 = identico)."""
    s1, s2 = normalizar(s1), normalizar(s2)
    if s1 == s2:
        return 1.0
    m, n = len(s1), len(s2)
    if m == 0 or n == 0:
        return 0.0
    # matriz de programacao dinamica
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev = dp[:]
        dp[0] = i
        for j in range(1, n + 1):
            custo = 0 if s1[i-1] == s2[j-1] else 1
            dp[j] = min(dp[j] + 1, dp[j-1] + 1, prev[j-1] + custo)
    distancia = dp[n]
    return 1 - distancia / max(m, n)


def cosseno(s1, s2):
    """Retorna similaridade cosseno entre 0 e 1."""
    s1, s2 = normalizar(s1), normalizar(s2)
    v1 = Counter(s1.split())
    v2 = Counter(s2.split())
    palavras = set(v1) | set(v2)
    if not palavras:
        return 0.0
    dot = sum(v1[p] * v2[p] for p in palavras)
    mag1 = math.sqrt(sum(v ** 2 for v in v1.values()))
    mag2 = math.sqrt(sum(v ** 2 for v in v2.values()))
    if mag1 == 0 or mag2 == 0:
        return 0.0
    return dot / (mag1 * mag2)


def similaridade_combinada(s1, s2, peso_lev=0.5, peso_cos=0.5):
    """Media ponderada entre Levenshtein e Cosseno."""
    return peso_lev * levenshtein(s1, s2) + peso_cos * cosseno(s1, s2)

testes = [
    ('oi', 'ola'),
    ('classificacao', 'classificação'),
    ('quem e verstappen', 'quem é o verstappen'),
    ('pizza', 'formula 1'),
]

print(f'{"Par":<40} {"Lev":>6} {"Cos":>6} {"Comb":>6}')
print('-' * 60)
for a, b in testes:
    lev = levenshtein(a, b)
    cos = cosseno(a, b)
    comb = similaridade_combinada(a, b)
    print(f'{a!r} x {b!r:<25} {lev:>6.2f} {cos:>6.2f} {comb:>6.2f}')

Par                                         Lev    Cos   Comb
------------------------------------------------------------
'oi' x 'ola'                       0.33   0.00   0.17
'classificacao' x 'classificação'             1.00   1.00   1.00
'quem e verstappen' x 'quem é o verstappen'       0.89   0.87   0.88
'pizza' x 'formula 1'                 0.11   0.00   0.06


In [12]:
LIMIAR = 0.35  # score minimo para aceitar uma intencao

def detectar_intencao(texto_usuario):
    melhor_intencao = 'desvio'
    melhor_score = 0.0
    melhor_padrao = ''

    for intencao, dados in CORPUS.items():
        for padrao in dados['padroes']:
            score = similaridade_combinada(texto_usuario, padrao)
            if score > melhor_score:
                melhor_score = score
                melhor_intencao = intencao
                melhor_padrao = padrao

    if melhor_score < LIMIAR:
        return 'desvio', melhor_score, melhor_padrao

    return melhor_intencao, melhor_score, melhor_padrao


# Teste
casos = [
    'oi',
    'quero saber sobre verstappen',
    'mostrar classificacao',
    'me fala das equipes',
    'qual e a receita de bolo',
    'tchau',
]

print(f'{"Input":<35} {"Intencao":<15} {"Score":>6} {"Padrao mais proximo"}')
print('-' * 85)
for caso in casos:
    intencao, score, padrao = detectar_intencao(caso)
    print(f'{caso:<35} {intencao:<15} {score:>6.2f}  {padrao}')

Input                               Intencao         Score Padrao mais proximo
-------------------------------------------------------------------------------------
oi                                  boas_vindas       1.00  oi
quero saber sobre verstappen        piloto            0.43  quem e verstappen
mostrar classificacao               consulta_csv      1.00  mostrar classificacao
me fala das equipes                 equipe            0.40  quais sao as equipes
qual e a receita de bolo            desvio            0.31  quem e verstappen
tchau                               despedida         1.00  tchau


In [13]:
DF_PILOTOS = pd.read_csv('pilotos_f1_2024.csv')

def consultar_csv(texto_usuario):
    texto = normalizar(texto_usuario)

    nomes = {
        'verstappen': 'Max Verstappen',
        'norris': 'Lando Norris',
        'leclerc': 'Charles Leclerc',
        'piastri': 'Oscar Piastri',
        'sainz': 'Carlos Sainz',
        'russell': 'George Russell',
        'hamilton': 'Lewis Hamilton',
        'perez': 'Sergio Perez',
        'alonso': 'Fernando Alonso',
        'hulkenberg': 'Nico Hulkenberg',
        'albon': 'Alexander Albon',
        'tsunoda': 'Yuki Tsunoda',
    }

    for chave, nome_completo in nomes.items():
        if chave in texto:
            linha = DF_PILOTOS[DF_PILOTOS['piloto'] == nome_completo].iloc[0]
            return (
                f"{linha['piloto']} ({linha['equipe']}) | "
                f"Pontos: {linha['pontos']} | "
                f"Vitorias: {linha['vitorias']} | "
                f"Podios: {linha['podios']} | "
                f"Poles: {linha['poles']}"
            )

    # sem nome especifico -> top 5
    top = DF_PILOTOS.sort_values('pontos', ascending=False).head(5)
    linhas = ['Top 5 campeonato 2024:']
    for i, (_, row) in enumerate(top.iterrows(), 1):
        linhas.append(f"  {i}. {row['piloto']} ({row['equipe']}) - {row['pontos']} pts")
    return '\n'.join(linhas)


# Teste
print(consultar_csv('dados de verstappen'))
print()
print(consultar_csv('mostrar classificacao'))
print()
print(consultar_csv('dados de hamilton'))

Max Verstappen (Red Bull) | Pontos: 437 | Vitorias: 9 | Podios: 14 | Poles: 9

Top 5 campeonato 2024:
  1. Max Verstappen (Red Bull) - 437 pts
  2. Lando Norris (McLaren) - 374 pts
  3. Charles Leclerc (Ferrari) - 356 pts
  4. Oscar Piastri (McLaren) - 292 pts
  5. Carlos Sainz (Ferrari) - 290 pts

Lewis Hamilton (Mercedes) | Pontos: 223 | Vitorias: 2 | Podios: 7 | Poles: 0


In [ ]:
GROQ_API_KEY = 'SUA_GROQ_API_KEY'
cliente_groq = Groq(api_key=GROQ_API_KEY)

SYSTEM_PROMPT = """Voce e o BotF1, assistente especialista em Formula 1.
Responda APENAS sobre Formula 1: pilotos, equipes, corridas, regras, historia e campeonatos.
Se a pergunta nao for sobre F1, diga que so fala sobre F1.
Respostas curtas e diretas, em portugues brasileiro, sem emojis."""

def responder_groq(mensagem_usuario, intencao, contexto_csv=None):
    conteudo = f"Intencao detectada: {intencao}\nPergunta: {mensagem_usuario}"
    if contexto_csv:
        conteudo += f"\nDados da base:\n{contexto_csv}"

    resposta = cliente_groq.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': conteudo}
        ],
        max_tokens=300,
        temperature=0.5
    )
    return resposta.choices[0].message.content.strip()


# Teste
print(responder_groq('oi, tudo bem?', 'boas_vindas'))
print()
print(responder_groq('quem e o verstappen?', 'piloto'))

In [ ]:
def chatbot():
    print('BotF1 iniciado. Digite sua pergunta (ou "tchau" para sair).')
    print('-' * 50)

    while True:
        try:
            entrada = input('Voce: ').strip()
        except (EOFError, KeyboardInterrupt):
            print('\nBot: Encerrando. Ate mais!')
            break

        if not entrada:
            continue

        intencao, score, _ = detectar_intencao(entrada)

        # despedida encerra o loop
        if intencao == 'despedida':
            print('Bot: Ate a proxima corrida!')
            break

        # consulta_csv: busca dados e passa como contexto pro Groq
        if intencao == 'consulta_csv':
            dados = consultar_csv(entrada)
            resposta = responder_groq(entrada, intencao, contexto_csv=dados)
        else:
            resposta = responder_groq(entrada, intencao)

        print(f'Bot [{intencao} | {score:.2f}]: {resposta}')
        print()

chatbot()

BotF1 iniciado. Digite sua pergunta (ou "tchau" para sair).
--------------------------------------------------


Voce:  quero saber sobre verstappen?


Bot [piloto | 0.43]: Max Verstappen é um piloto holandês que atualmente dirige pela equipe Red Bull Racing. Ele é filho do ex-piloto Jos Verstappen e já conquistou o Campeonato Mundial de Pilotos em 2021 e 2022.



Voce:  qual e a receita de bolo


Bot [desvio | 0.31]: Só falo sobre F1. Não tenho conhecimento sobre receita de bolo.



Voce:  mostrar classificacao


Bot [consulta_csv | 1.00]: A classificação do campeonato de 2024 é:
1. Max Verstappen (Red Bull) - 437 pontos
2. Lando Norris (McLaren) - 374 pontos
3. Charles Leclerc (Ferrari) - 356 pontos
4. Oscar Piastri (McLaren) - 292 pontos
5. Carlos Sainz (Ferrari) - 290 pontos



Voce:  dados de hamilton


Bot [consulta_csv | 1.00]: Lewis Hamilton é um piloto britânico que correu pela equipe Mercedes. Ele conquistou 223 pontos, 2 vitórias, 7 podios e 0 poles position em sua carreira.

